# Normalization & adjusted EBITDA

Walk from **reported EBITDA** to **adjusted EBITDA** using `NormalizationConfig` with fixed add-backs, **percentage-of-revenue** fees, and **capped** synergies.

## Concept

`NormalizationConfig` is JSON-serializable: each adjustment has a `type` of `fixed` (per-period map), `percentage_of_node`, or `formula`. Caps reference a **base node** and a **percentage limit**. Python exposes `normalize(result, config)` which returns a JSON string; parse it with `json.loads` to get Python dict rows.

**Cap semantics:** When an adjustment has a `cap`, the limit is computed as `cap.value * running_adjusted_value` — meaning the cap is a percentage of the **cumulative adjusted total** (base + all prior adjustments), not the raw base value. This prevents runaway add-backs while staying transparent in the output.

In [1]:
import json

from finstack.statements import (
    Evaluator,
    ModelBuilder,
    NormalizationConfig,
    normalize,
)

PERIODS = ["2025Q1", "2025Q2"]


def series(v):
    return list(zip(PERIODS, v))


b = ModelBuilder("norm-demo")
b.periods("2025Q1..Q2", None)
b.value("revenue", series([1000.0, 1100.0]))
b.value("ebitda", series([120.0, 135.0]))
spec = b.build()
res = Evaluator().evaluate(spec)

cfg = NormalizationConfig("ebitda")
print("Empty config:", cfg, "| adjustments:", cfg.adjustment_count)


Loaded finstack.statements + finstack.statements_analytics
Empty config: NormalizationConfig(target="ebitda", adjustments=0) | adjustments: 0


In [2]:
print("Normalization JSON config + engine")
cfg_json = {
    "target_node": "ebitda",
    "adjustments": [
        {
            "id": "restructuring",
            "name": "Restructuring add-back",
            "value": {
                "type": "fixed",
                "amounts": {"2025Q1": 8.0, "2025Q2": 5.0},
            },
        },
        {
            "id": "mgmt_fee",
            "name": "Mgmt fee (% of revenue)",
            "value": {"type": "percentage_of_node", "node_id": "revenue", "percentage": 0.02},
        },
        {
            "id": "synergies",
            "name": "Synergies (capped at 15% of running adjusted EBITDA)",
            "value": {
                "type": "fixed",
                "amounts": {"2025Q1": 30.0, "2025Q2": 25.0},
            },
            "cap": {"base_node": "ebitda", "value": 0.15},
        },
    ],
}
full_cfg = NormalizationConfig.from_json(json.dumps(cfg_json))
print("Loaded adjustments:", full_cfg.adjustment_count)

rows = json.loads(normalize(res, full_cfg))
for row in rows:
    print(f"\n{row['period']}:  base={row['base_value']:.1f}  final={row['final_value']:.1f}")
    for adj in row["adjustments"]:
        capped = " (CAPPED)" if adj["is_capped"] else ""
        print(f"  + {adj['name']:45s}  raw={adj['raw_amount']:6.1f}  applied={adj['capped_amount']:6.1f}{capped}")


Normalization JSON config + engine
Loaded adjustments: 3
normalize JSON: [{"period":"2025Q1","base_value":120.0,"adjustments":[{"adjustment_id":"restructuring","name":"Restructuring add-back","raw_amount":8.0,"capped_amount":8.0,"is_capped":false},{"adjustment_id":"mgmt_fee","name":"Mgmt fee (% of revenue)","raw_amount":20.0,"capped_amount":20.0,"is_capped":false},{"adjustment_id":"synergies","name":"Synergies (capped at 15% of EBITDA)","raw_amount":30.0,"capped_amount":22.2,"is_capped":true}],"final_value":170.2},{"period":"2025Q2","base_value":135.0,"adjustments":[{"adjustment_id":"restructuring","name":"Restructuring add-back","raw_amount":5.0,"capped_amount":5.0,"is_capped":false},{"adjustment_id":"mgmt_fee","name":"Mgmt fee (% of revenue)","raw_amount":22.0,"capped_amount":22.0,"is_capped":false},{"adjustment_id":"synergies","name":"Synergies (capped at 15% of EBITDA)","raw_amount":25.0,"capped_amount":24.3,"is_capped":true}],"final_value":186.3}]
period 2025Q1 base 120.0 final 17

## Takeaways

- Keep a **machine-readable adjustment policy** (`from_json`) next to human memos for audit.
- **Percentage fees** and **caps** prevent runaway add-backs while staying transparent in the per-period breakdown returned by `normalize`.
- Merge normalized series back into models with downstream nodes if you need **forecast adjusted EBITDA**.